# 🪥 Notebook 1: When You Don't Detect Corruption

Computers are not perfect. The bytes your program *writes* are not always the same bytes it *reads* back later:

- **Disks bit-rot.** A magnetic bit can flip after years on a platter, or an SSD cell can lose charge.
- **Networks flip bits.** Cables, routers, and wireless links occasionally corrupt data. TCP/IP has its *own* checksum, but it is weak (16 bits), and anything above TCP — app code, proxies, middleboxes — can still hand your program bad bytes.
- **RAM has cosmic-ray errors.** A bit can flip for no reason; ECC memory in servers exists precisely because of this.
- **Bugs corrupt data.** A half-written file after a crash, a truncated upload, an off-by-one in a copy loop.

If your code just trusts whatever bytes come back, it will *silently* propagate corruption to users, to replicas, and to backups. That is the nightmare: the bad data looks fine, and by the time anyone notices, every copy is already wrong.

A **checksum** is a short fingerprint of the data (usually 4–32 bytes). You compute it when you write, store it alongside the data, and recompute it when you read. If the fingerprints don't match — the data is corrupt, and you *refuse to use it* instead of pretending everything is fine.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/checksum
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 BAD: store raw bytes, trust them blindly

We will simulate a single-bit flip on disk (think: cosmic ray on an SSD cell, or a cheap SATA cable). Our program reads the file and happily returns the wrong answer.

In [1]:
import os, tempfile

WORKDIR = tempfile.mkdtemp(prefix='nochk_')
PATH = os.path.join(WORKDIR, 'account.bin')

# Pretend this is a row we wrote to disk: Alice's balance is $100.
original = b'account=alice; balance=100'
with open(PATH, 'wb') as f:
    f.write(original)

# -- Simulate a single-bit flip somewhere in the file (bit-rot / bad cable / bad RAM).
raw = bytearray(open(PATH, 'rb').read())
victim = raw.index(b'100')  # find the '1' in '100'
raw[victim] ^= 0b0000_0100  # flip one bit inside '1' -> '5'
with open(PATH, 'wb') as f:
    f.write(raw)

# -- Our application reads the file back, unaware anything is wrong.
loaded = open(PATH, 'rb').read()
print('we wrote :', original)
print('we read  :', loaded)
print()
print('Only ONE bit flipped, but the app now believes Alice has $500.')
print('Nothing raised an error. This is silent corruption.')


we wrote : b'account=alice; balance=100'
we read  : b'account=alice; balance=500'

Only ONE bit flipped, but the app now believes Alice has $500.
Nothing raised an error. This is silent corruption.


### Why this is dangerous in a distributed system

One corrupt row by itself is bad. In a distributed system it is *much worse*:

1. A leader reads the row and sends it to two replicas.
2. Replicas store the corrupt row. Now **every copy is wrong** and no replica disagrees.
3. The nightly backup job copies the corrupt row to the backup bucket.
4. A month later someone notices — but all 30 daily backups already contain the bad value.

The fix is simple: **every piece of data carries a checksum**, and we check it every time we read. If it mismatches, we stop — we do not replicate, we do not back up, we raise an alarm.

👉 Notebook 2 adds checksums and shows how CRC32, MD5, and SHA-256 compare.